# Train ADR trên Google Colab

1. Trên máy: chạy `python scripts/export_colab.py`.
2. Chọn runtime có GPU nếu khả dụng.
3. Upload `results/ADR_Product_colab.zip` ở cell dưới.
4. Chạy lần lượt các cell; tải và giải nén artifact về project local.

Dataset SIDER công khai được tải tự động. Chỉ evaluate validation, không tự mở tập test.


In [ ]:
from google.colab import files
from io import BytesIO
from zipfile import ZipFile

uploaded = files.upload()
name = "ADR_Product_colab.zip"
if name not in uploaded:
    raise ValueError(f"Hãy chọn {name}")
with ZipFile(BytesIO(uploaded[name])) as archive:
    archive.extractall("/content")


In [ ]:
%cd /content/ADR_Product
%pip install -e . pytest


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## Training

Mặc định tối đa 50 epoch; early stopping theo validation macro AP. `auto` dùng CUDA khi có, nếu không dùng CPU. Chỉ muốn thử pipeline thì thêm `--epochs 2` vào danh sách lệnh.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/run_pipeline.py", "--models", "mlp", "cnn", "--device", "auto"], check=True)


In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("results/run_summary.json").read_text())
for name, result in summary.items():
    print(name, "validation macro AP:", result["val_macro_ap"], "artifact:", result["run_dir"])


## Lưu kết quả

Runtime Colab có thể kết thúc; tải model về ngay sau khi train. ZIP chứa thư mục `artifacts/`; giải nén vào thư mục gốc project rồi chạy lại Streamlit. Muốn evaluate tại máy khác, mang kèm đúng dataset/split đã train và dùng `--base` để chỉ đường dẫn mới.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/adr_trained_artifacts", "zip", root_dir=".", base_dir="artifacts")
files.download(archive)


Nguồn: [Colab FAQ](https://research.google.com/colaboratory/faq.html). GPU và thời lượng runtime tùy tài nguyên khả dụng.
